# Resist Depth Validation and 3D Demo

This notebook uses 2D x-depth heatmaps for validation. The Beer-Lambert-only stack and the focus-depth coupled stack are shown separately so their model assumptions are not conflated.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src.aerial import aerial_image
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.pupil import PupilSpec
from src.resist_depth import depth_resolved_dose_stack, focus_depth_resolved_resist

## Study Mask, Aerial Image, and Depth Sweep

The depth sweep uses 31 samples. This is enough for a smooth x-depth validation heatmap while staying fast for headless notebook execution.

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
mask = kirchhoff_mask(line_space_pattern(grid, pitch_m=40e-9))
pupil_spec = PupilSpec(grid_size=256)
aerial, wafer = aerial_image(mask, grid, pupil_spec=pupil_spec, anamorphic=False)
depths = np.linspace(0.0, 60e-9, 31)

print("aerial shape:", aerial.shape)
print("aerial min/max:", float(np.min(aerial)), float(np.max(aerial)))
print("depth samples:", len(depths))

## 2D Validation: Beer-Lambert Attenuation Only

This model broadcasts one aerial image through depth and applies exponential attenuation. Pattern shape should remain stable while mean dose decreases with depth.

In [ ]:
dose_stack = depth_resolved_dose_stack(
    aerial,
    depths,
    dose=1.0,
    absorption_coefficient_m_inv=2.0e6,
)

x_nm = (np.arange(wafer.nx) - wafer.nx / 2) * wafer.pixel_x_m * 1e9
depth_nm = depths * 1e9
center_y = dose_stack.shape[1] // 2
dose_x_depth = dose_stack[:, center_y, :]
top_mean = float(np.mean(dose_stack[0]))
bottom_mean = float(np.mean(dose_stack[-1]))

print("dose_stack shape:", dose_stack.shape)
print("dose_stack min/max:", float(np.min(dose_stack)), float(np.max(dose_stack)))
print("dose_stack finite:", bool(np.all(np.isfinite(dose_stack))))
print("depth samples:", len(depths))
print("top mean dose:", top_mean)
print("bottom mean dose:", bottom_mean)
print("bottom/top mean dose:", bottom_mean / max(top_mean, 1e-12))

plt.figure(figsize=(9, 5))
plt.imshow(
    dose_x_depth,
    extent=[x_nm.min(), x_nm.max(), depth_nm.max(), depth_nm.min()],
    aspect="auto",
)
plt.xlabel("x [nm]")
plt.ylabel("depth [nm]")
plt.title("2D depth-resolved dose, Beer-Lambert attenuation only")
plt.colorbar(label="dose")
plt.tight_layout()
plt.show()

## 2D Validation: Focus-Depth Coupled Resist

This model recomputes the aerial image at each depth-coupled defocus slice before applying dose attenuation and thresholding.

In [ ]:
focus_coupled = focus_depth_resolved_resist(
    mask,
    grid,
    depths,
    pupil_spec=PupilSpec(grid_size=256),
    top_defocus_m=0.0,
    dose=1.0,
    threshold=0.3,
    absorption_coefficient_m_inv=2.0e6,
    anamorphic=False,
)

dose_stack_fc = focus_coupled.dose_stack
center_y_fc = dose_stack_fc.shape[1] // 2
dose_x_depth_fc = dose_stack_fc[:, center_y_fc, :]
x_nm_fc = (
    np.arange(focus_coupled.wafer.nx) - focus_coupled.wafer.nx / 2
) * focus_coupled.wafer.pixel_x_m * 1e9
depth_nm_fc = np.asarray(focus_coupled.depth_values_m) * 1e9
top_mean_fc = float(np.mean(dose_stack_fc[0]))
bottom_mean_fc = float(np.mean(dose_stack_fc[-1]))

print("focus-coupled dose_stack shape:", dose_stack_fc.shape)
print("focus-coupled dose_stack min/max:", float(np.min(dose_stack_fc)), float(np.max(dose_stack_fc)))
print("focus-coupled dose_stack finite:", bool(np.all(np.isfinite(dose_stack_fc))))
print("top mean dose:", top_mean_fc)
print("bottom mean dose:", bottom_mean_fc)
print("bottom/top mean dose:", bottom_mean_fc / max(top_mean_fc, 1e-12))

plt.figure(figsize=(9, 5))
plt.imshow(
    dose_x_depth_fc,
    extent=[x_nm_fc.min(), x_nm_fc.max(), depth_nm_fc.max(), depth_nm_fc.min()],
    aspect="auto",
)
plt.xlabel("x [nm]")
plt.ylabel("depth [nm]")
plt.title("2D depth-resolved dose with focus-depth coupling")
plt.colorbar(label="dose")
plt.tight_layout()
plt.show()

## Demo Only: 3D Resist Dose Surface

This surface reuses the Beer-Lambert stack. Use it for qualitative presentation only; use the heatmaps above for validation.

In [ ]:
X, Z = np.meshgrid(x_nm, depth_nm)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(
    X,
    Z,
    dose_x_depth,
    cmap="magma",
    linewidth=0,
    antialiased=True,
    alpha=0.9,
)
ax.set_xlabel("x [nm]")
ax.set_ylabel("depth [nm]")
ax.set_zlabel("dose")
ax.set_title("Demo only: 3D Beer-Lambert dose surface; validate with 2D plots above")
fig.colorbar(surface, ax=ax, shrink=0.6, pad=0.1)
fig.tight_layout()
plt.show()